In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

In [2]:
# =========================================================
# 1. SYNTHETIC BANK CUSTOMER CHURN DATASET
# =========================================================
def generate_bank_churn_data(n=50000, seed=42):
    """
    Generate a synthetic bank customer churn dataset with a heavily imbalanced
    true churn rate. The dataset is designed for demonstrating imbalance,
    downsampling, model training, and Bayesian prior correction.
    """
    rng = np.random.default_rng(seed)

    customer_id = np.arange(100000, 100000 + n)

    age = rng.integers(21, 76, n)
    gender = rng.choice(["Male", "Female"], n, p=[0.52, 0.48])
    region = rng.choice(
        ["Ontario", "British Columbia", "Alberta", "Quebec"],
        n,
        p=[0.40, 0.20, 0.20, 0.20]
    )

    tenure_years = rng.integers(0, 16, n)
    annual_income = np.clip(rng.normal(85000, 30000, n), 25000, 250000)
    credit_score = np.clip(rng.normal(690, 85, n), 300, 850).round().astype(int)

    account_balance = np.clip(rng.normal(42000, 30000, n), 0, 300000)
    monthly_salary_deposit = np.clip(annual_income / 12 + rng.normal(0, 600, n), 0, 30000)
    avg_monthly_spend = np.clip(rng.normal(2600, 1400, n), 50, 20000)

    num_products = rng.choice([1, 2, 3, 4], n, p=[0.38, 0.37, 0.20, 0.05])

    has_credit_card = rng.choice([0, 1], n, p=[0.18, 0.82])
    has_mortgage = rng.choice([0, 1], n, p=[0.70, 0.30])
    has_personal_loan = rng.choice([0, 1], n, p=[0.82, 0.18])
    has_investment_account = rng.choice([0, 1], n, p=[0.72, 0.28])

    credit_card_monthly_usage = np.where(
        has_credit_card == 1,
        np.clip(rng.normal(1900, 1100, n), 0, 15000),
        0
    )

    mobile_app_logins_per_month = rng.integers(0, 45, n)
    online_banking_logins_per_month = rng.integers(0, 35, n)
    branch_visits_per_quarter = rng.integers(0, 9, n)

    customer_service_calls = rng.poisson(1.8, n)
    complaints_last_12m = rng.poisson(0.35, n)
    late_payment_count = rng.poisson(0.30, n)

    balance_change_6m_pct = np.clip(rng.normal(1, 12, n), -70, 70)
    transaction_count_monthly = np.clip(rng.normal(30, 13, n), 1, 140).round().astype(int)

    digital_engagement_score = (
        0.65 * mobile_app_logins_per_month +
        0.35 * online_banking_logins_per_month
    )

    # Latent churn score
    # Coefficients are chosen so a reasonable signal exists
    linear_score = (
        -0.10 * tenure_years
        -0.25 * num_products
        +0.70 * complaints_last_12m
        +0.55 * late_payment_count
        +0.12 * customer_service_calls
        -0.025 * mobile_app_logins_per_month
        -0.015 * online_banking_logins_per_month
        -0.000012 * account_balance
        -0.000010 * credit_card_monthly_usage
        -0.18 * has_credit_card
        -0.22 * has_investment_account
        -0.12 * has_mortgage
        -0.025 * balance_change_6m_pct
        -0.0018 * (credit_score - 650)
        +0.35 * (avg_monthly_spend > (monthly_salary_deposit * 0.9)).astype(int)
        +0.18 * (num_products == 1).astype(int)
        +rng.normal(0, 0.9, n)
    )

    # Intercept chosen to create a true imbalanced churn problem
    intercept = -3.3
    raw_prob = 1 / (1 + np.exp(-(intercept + linear_score)))
    churn = rng.binomial(1, raw_prob, n)

    df = pd.DataFrame({
        "customer_id": customer_id,
        "age": age,
        "gender": gender,
        "region": region,
        "tenure_years": tenure_years,
        "annual_income": np.round(annual_income, 2),
        "credit_score": credit_score,
        "account_balance": np.round(account_balance, 2),
        "monthly_salary_deposit": np.round(monthly_salary_deposit, 2),
        "avg_monthly_spend": np.round(avg_monthly_spend, 2),
        "num_products": num_products,
        "has_credit_card": has_credit_card,
        "has_mortgage": has_mortgage,
        "has_personal_loan": has_personal_loan,
        "has_investment_account": has_investment_account,
        "credit_card_monthly_usage": np.round(credit_card_monthly_usage, 2),
        "mobile_app_logins_per_month": mobile_app_logins_per_month,
        "online_banking_logins_per_month": online_banking_logins_per_month,
        "branch_visits_per_quarter": branch_visits_per_quarter,
        "customer_service_calls": customer_service_calls,
        "complaints_last_12m": complaints_last_12m,
        "late_payment_count": late_payment_count,
        "balance_change_6m_pct": np.round(balance_change_6m_pct, 2),
        "transaction_count_monthly": transaction_count_monthly,
        "digital_engagement_score": np.round(digital_engagement_score, 2),
        "churn": churn
    })

    return df


# =========================================================
# 2. CHECK CLASS DISTRIBUTION
# =========================================================
def class_distribution(y, label="Dataset"):
    """
    Print class counts and class ratio.
    """
    counts = pd.Series(y).value_counts().sort_index()
    proportions = pd.Series(y).value_counts(normalize=True).sort_index()

    print(f"\n{label} distribution")
    print("-" * 50)
    print("Counts:")
    print(counts)
    print("\nProportions:")
    print(proportions)
    print(f"\nChurn rate: {proportions.get(1, 0):.4%}")


# =========================================================
# 3. DOWNSAMPLE TRAINING DATA TO 9:1
# =========================================================
def downsample_to_ratio(X, y, majority_to_minority_ratio=9, random_state=42):
    """
    Downsample the majority class so that majority:minority becomes 9:1.

    Example:
    If minority class has 500 rows, majority class will be downsampled to 4500.
    """
    df = X.copy()
    df["target"] = y.values if isinstance(y, pd.Series) else y

    minority = df[df["target"] == 1]
    majority = df[df["target"] == 0]

    minority_count = len(minority)
    majority_target_count = majority_to_minority_ratio * minority_count

    majority_downsampled = majority.sample(
        n=min(majority_target_count, len(majority)),
        random_state=random_state,
        replace=False
    )

    sampled = pd.concat([minority, majority_downsampled], axis=0)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    X_sampled = sampled.drop(columns=["target"])
    y_sampled = sampled["target"]

    return X_sampled, y_sampled


# =========================================================
# 4. PREPROCESSOR + MODEL
# =========================================================
def build_logistic_model(X):
    """
    Build a preprocessing + logistic regression pipeline.
    Logistic regression is useful here because it produces probabilities
    that can then be corrected using Bayes prior adjustment.
    """
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ]
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ])

    return model


# =========================================================
# 5. BAYESIAN PRIOR CORRECTION
# =========================================================
def adjust_probabilities_for_prior_shift(p_sample, pi_sample, pi_true, eps=1e-12):
    """
    Adjust predicted probabilities from a model trained on sampled data
    to the probabilities under the true population prior.

    Parameters
    ----------
    p_sample : array-like
        Predicted probabilities from the model trained on the sampled data.
    pi_sample : float
        Positive class rate in sampled training data.
    pi_true : float
        Positive class rate in the true population.
    eps : float
        Small value to avoid division by zero.

    Returns
    -------
    np.ndarray
        Prior-corrected probabilities.
    """
    p_sample = np.clip(np.asarray(p_sample), eps, 1 - eps)

    odds_sample = p_sample / (1 - p_sample)
    prior_odds_ratio = (pi_true / (1 - pi_true)) / (pi_sample / (1 - pi_sample))

    odds_true = odds_sample * prior_odds_ratio
    p_true = odds_true / (1 + odds_true)

    return p_true


# =========================================================
# 6. THRESHOLD HELPER
# =========================================================
def best_f1_threshold(y_true, y_prob):
    """
    Find threshold with the best F1 score using precision-recall curve.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    # thresholds has length n-1, precision/recall have length n
    precision = precision[:-1]
    recall = recall[:-1]

    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-12)
    idx = np.argmax(f1_scores)

    return thresholds[idx], f1_scores[idx]


# =========================================================
# 7. EVALUATION FUNCTION
# =========================================================
def evaluate_predictions(y_true, y_prob, label="Model", threshold=0.5):
    """
    Evaluate probabilities using AUC, PR AUC, confusion matrix, and classification report.
    """
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{label}")
    print("=" * 70)
    print(f"Threshold used: {threshold:.4f}")
    print(f"ROC AUC: {roc_auc_score(y_true, y_prob):.4f}")
    print(f"PR AUC : {average_precision_score(y_true, y_prob):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))



In [6]:

# =========================================================
# 8. FULL END-TO-END PIPELINE
# =========================================================
def run_churn_model_demo(
    n=50000,
    test_size=0.30,
    sampled_ratio=9,
    random_state=42,
    save_csv=True
):
    """
    Full demonstration:
    1. Generate highly imbalanced bank churn data
    2. Split train/test on original data
    3. Downsample train data to 9:1
    4. Train logistic churn model
    5. Predict on original test set
    6. Adjust probabilities back to true population prior using Bayes formula
    7. Evaluate before and after correction
    """
    # -------------------------
    # Step 1: Generate data
    # -------------------------
    df = generate_bank_churn_data(n=n, seed=random_state)

    #if save_csv:
    #    df.to_csv("bank_customer_churn_actual_imbalanced.csv", index=False)
    #    print("\nSaved: bank_customer_churn_actual_imbalanced.csv")

    X = df.drop(columns=["customer_id", "churn"])
    y = df["churn"]

    class_distribution(y, label="True full dataset")


In [13]:
outputs = run_churn_model_demo()
df = outputs


True full dataset distribution
--------------------------------------------------
Counts:
churn
0    49513
1      487
Name: count, dtype: int64

Proportions:
churn
0    0.99026
1    0.00974
Name: proportion, dtype: float64

Churn rate: 0.9740%


In [14]:
    # -------------------------
    # Step 2: Train/test split on true data
    # -------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    class_distribution(y_train, label="Original train set")
    class_distribution(y_test, label="Original test set")

    true_train_prior = y_train.mean()

    # -------------------------
    # Step 3: Downsample to 9:1 in train set
    # -------------------------
    X_train_sampled, y_train_sampled = downsample_to_ratio(
        X_train, y_train,
        majority_to_minority_ratio=sampled_ratio,
        random_state=random_state
    )

    sampled_train_prior = y_train_sampled.mean()

    class_distribution(y_train_sampled, label=f"Downsampled training set ({sampled_ratio}:1)")

    train_sampled_df = X_train_sampled.copy()
    train_sampled_df["churn"] = y_train_sampled.values
    if save_csv:
        train_sampled_df.to_csv("bank_customer_churn_train_9_to_1.csv", index=False)
        print("Saved: bank_customer_churn_train_9_to_1.csv")

    # -------------------------
    # Step 4: Build and train model
    # -------------------------
    model = build_logistic_model(X_train_sampled)
    model.fit(X_train_sampled, y_train_sampled)

    # -------------------------
    # Step 5: Predict on test set
    # -------------------------
    p_sample = model.predict_proba(X_test)[:, 1]

    # -------------------------
    # Step 6: Bayesian prior correction
    # -------------------------
    p_corrected = adjust_probabilities_for_prior_shift(
        p_sample=p_sample,
        pi_sample=sampled_train_prior,
        pi_true=true_train_prior
    )

    # -------------------------
    # Step 7: Evaluate
    # -------------------------
    print("\n" + "#" * 70)
    print("PRIOR INFORMATION")
    print("#" * 70)
    print(f"True train churn prior     : {true_train_prior:.6f}")
    print(f"Sampled train churn prior  : {sampled_train_prior:.6f}")

    # Default threshold
    evaluate_predictions(
        y_true=y_test,
        y_prob=p_sample,
        label="Model probabilities from sampled 9:1 training data (uncorrected)",
        threshold=0.5
    )

    evaluate_predictions(
        y_true=y_test,
        y_prob=p_corrected,
        label="Model probabilities after Bayesian prior correction",
        threshold=0.5
    )

    # Optional: better threshold for corrected probabilities
    best_threshold, best_f1 = best_f1_threshold(y_test, p_corrected)
    print(f"\nBest threshold on corrected probabilities by F1: {best_threshold:.4f}")
    print(f"Best F1 at that threshold: {best_f1:.4f}")

    evaluate_predictions(
        y_true=y_test,
        y_prob=p_corrected,
        label="Bayesian-corrected probabilities with F1-optimized threshold",
        threshold=best_threshold
    )

    # -------------------------
    # Step 8: Return useful outputs
    # -------------------------
    results = X_test.copy()
    results["actual_churn"] = y_test.values
    results["pred_prob_sampled"] = np.round(p_sample, 6)
    results["pred_prob_corrected"] = np.round(p_corrected, 6)
    results["pred_class_corrected_0_5"] = (p_corrected >= 0.5).astype(int)
    results["pred_class_corrected_best_f1"] = (p_corrected >= best_threshold).astype(int)

    if save_csv:
        results.to_csv("bank_customer_churn_scored_test_set.csv", index=False)
        print("\nSaved: bank_customer_churn_scored_test_set.csv")

    return {
        "full_data": df,
        "X_train_sampled": X_train_sampled,
        "y_train_sampled": y_train_sampled,
        "model": model,
        "scored_test": results,
        "true_train_prior": true_train_prior,
        "sampled_train_prior": sampled_train_prior,
        "best_threshold": best_threshold
    }


# =========================================================
# 9. MAIN
# =========================================================
if __name__ == "__main__":
    outputs = run_churn_model_demo(
        n=50000,
        test_size=0.30,
        sampled_ratio=9,
        random_state=42,
        save_csv=True
    )

IndentationError: expected an indented block after 'if' statement on line 120 (3825725366.py, line 121)